# CHEM 269 — Tier-2 xtb+GBSA: 5 Reference Compounds

**Alternative to CREST+ALPB.** Uses GFN2-xTB geometry optimization with GBSA implicit solvation in two dielectric environments to compute dual-dielectric ΔPSA on 5 reference compounds.

## Scientific framing
Instead of CREST's exhaustive conformer ensemble (iMTD-GC), this notebook runs a **single gradient-descent optimization** in each solvent:
- `xtb --opt --gbsa water` → geometry optimized for aqueous environment (ε≈80)
- `xtb --opt --gbsa chcl3` → geometry optimized for membrane-mimetic environment (ε≈4.8)

The optimized geometry in each solvent is then used to compute 3D PSA (identical to Tier-1 protocol). ΔPSA = PSA(water) − PSA(CHCl3) captures the conformational switch driven by solvation.

**This is a valid Tier-2 approach because:**
- GFN2-xTB with GBSA is the same underlying physics as CREST uses
- Single-structure optimization finds the local-minimum geometry in each environment
- Differences reflect real solvation-driven conformational changes
- ~3 min/compound vs hours for CREST

**Reference compounds:**
- **CsA** — gold-standard chameleon, lit. ΔPSA ~75 Å² (Witek JCTC 2016)
- **1NMe3** — N-methylated hexapeptide, permeable
- **HexPep** — parent hexapeptide, impermeable
- **DP172** — highly permeable pharmaceutical
- **c*[PSLYF]** — impermeable control

## Setup
1. `Runtime → Change runtime type` → CPU runtime is fine (no GPU needed)
2. Run Cell 1 (installs condacolab, runtime restarts automatically)
3. After restart, skip to Cell 2
4. Run all remaining cells top to bottom

## Output
- `tier2_xtb_results.csv` — xtb+GBSA ΔPSA, ΔHB, shape descriptors for all 5 compounds
- `tier2_xtb_summary.txt` — comparison vs Tier-1 and literature values

In [ ]:
# ── CELL 1: Install condacolab (runtime restarts automatically) ────────────────
# After the restart, skip back to Cell 2.
try:
    import condacolab
    print('condacolab already installed — skip to Cell 2')
except ImportError:
    !pip install -q condacolab
    import condacolab
    condacolab.install()

In [ ]:
# ── CELL 2: Install xtb + RDKit ───────────────────────────────────────────────
# xtb provides GFN2-xTB + GBSA solvation. No CREST needed.
import subprocess, sys, os

os.environ['OPENBLAS_NUM_THREADS'] = '1'

print('Installing xtb via conda...')
subprocess.run(
    ['conda', 'install', 'conda-forge::xtb', '-y'],
    check=True
)

print('Installing RDKit via pip...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'rdkit'],
    check=True
)

print('Cleaning conda cache...')
subprocess.run(['conda', 'clean', '--all', '-y'], check=True)

import rdkit
r = subprocess.run(['xtb', '--version'], capture_output=True, text=True)
print('xtb:  ', r.stdout.strip()[:80] or r.stderr.strip()[:80])
print('RDKit:', rdkit.__version__)

import multiprocessing, psutil
print(f'CPUs : {multiprocessing.cpu_count()}')
print(f'RAM  : {psutil.virtual_memory().total/1e9:.0f} GB')

In [ ]:
# ── CELL 3: Mount Google Drive (optional) ─────────────────────────────────────
USE_DRIVE = True   # set False to save locally only

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = '/content/drive/MyDrive/chem269_tier2_xtb/'
    import os; os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f'Drive mounted. Results → {RESULTS_DIR}')
else:
    RESULTS_DIR = '/content/tier2_xtb_results/'
    import os; os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f'Drive skipped. Results → {RESULTS_DIR}')

RESULTS_CSV     = RESULTS_DIR + 'tier2_xtb_results.csv'
RESULTS_SUMMARY = RESULTS_DIR + 'tier2_xtb_summary.txt'
WORK_ROOT       = '/tmp/xtb_runs'
os.makedirs(WORK_ROOT, exist_ok=True)
print(f'Results CSV → {RESULTS_CSV}')

In [ ]:
# ── CELL 4: Helper functions ───────────────────────────────────────────────────
import os, subprocess, shutil, time
import numpy as np
from pathlib import Path
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors3D, rdFreeSASA
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Geometry import rdGeometry

RDLogger.DisableLog('rdApp.*')
os.environ['OMP_NUM_THREADS'] = '2'   # xtb uses OMP internally

_BONDI  = {'H':1.20,'C':1.70,'N':1.55,'O':1.52,'S':1.80,'P':1.80,
            'F':1.47,'Cl':1.75,'Br':1.85,'I':1.98}
_POLAR  = {'N','O','S','P'}
HB_DONOR    = Chem.MolFromSmarts('[N,O;!H0]')
HB_ACCEPTOR = Chem.MolFromSmarts('[N,O]')


def smiles_to_xyz(smiles, mol_id, work_dir):
    """SMILES → ETKDGv3 + MMFF94s starting geometry → XYZ file."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None
    try:
        mol = rdMolStandardize.FragmentParent(mol)
        Chem.SanitizeMol(mol)
    except Exception:
        pass
    mol_h = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    params.useMacrocycleTorsions = True
    params.useSmallRingTorsions  = True
    params.maxIterations = 2000
    if AllChem.EmbedMolecule(mol_h, params) != 0:
        return None, None
    AllChem.MMFFOptimizeMolecule(mol_h, mmffVariant='MMFF94s', maxIters=2000)
    conf = mol_h.GetConformer()
    lines = [str(mol_h.GetNumAtoms()), f'ID={mol_id}']
    for atom in mol_h.GetAtoms():
        p = conf.GetAtomPosition(atom.GetIdx())
        lines.append(f'{atom.GetSymbol()}  {p.x:.6f}  {p.y:.6f}  {p.z:.6f}')
    xyz_path = Path(work_dir) / f'{mol_id}_start.xyz'
    xyz_path.write_text('\n'.join(lines))
    return xyz_path, mol_h


def run_xtb_opt(xyz_path, solvent, run_dir):
    """
    Run: xtb <input.xyz> --opt --gbsa <solvent> --gfn 2
    Returns path to optimized XYZ (xtbopt.xyz) or None on failure.

    GFN2-xTB + GBSA notes:
    - GFN2-xTB is the default semiempirical method (same as CREST uses internally)
    - GBSA (generalized Born with surface area) provides implicit solvation
    - Supported solvents: water, chcl3, acetone, dmso, methanol, thf, toluene, etc.
    - Optimization uses the BFGS L-BFGS algorithm; convergence ~50-200 steps
    - Output: xtbopt.xyz (optimized geometry), xtbopt.log (optimization trajectory)
    - Typical wall time: 1-5 min per cyclic peptide on CPU
    """
    solvent_dir = Path(run_dir) / solvent
    shutil.rmtree(solvent_dir, ignore_errors=True)
    solvent_dir.mkdir(parents=True, exist_ok=True)

    cmd = ['xtb', str(xyz_path.resolve()), '--opt', '--gbsa', solvent, '--gfn', '2']
    log_path = solvent_dir / 'xtb.log'

    try:
        with open(log_path, 'w') as lf:
            r = subprocess.run(
                cmd,
                cwd=str(solvent_dir),
                stdout=lf,
                stderr=subprocess.STDOUT,
                timeout=1800   # 30 min hard limit per compound per solvent
            )
        opt_xyz = solvent_dir / 'xtbopt.xyz'
        if r.returncode != 0:
            print(f'    xtb non-zero exit ({r.returncode}) — check {log_path}')
            return None
        if not opt_xyz.exists():
            print(f'    xtbopt.xyz not found — check {log_path}')
            return None
        return opt_xyz
    except subprocess.TimeoutExpired:
        print(f'    xtb timeout (30 min) for solvent={solvent}')
        return None
    except FileNotFoundError:
        print('    xtb binary not found — run Cell 2 first')
        return None


def xyz_to_mol(xyz_path, template_mol):
    """Load xtbopt.xyz geometry onto RDKit template mol (preserves bond topology)."""
    lines = Path(xyz_path).read_text().strip().split('\n')
    try:
        n = int(lines[0].strip())
    except ValueError:
        return None
    if n != template_mol.GetNumAtoms():
        print(f'    Atom count mismatch: xtb={n}, template={template_mol.GetNumAtoms()}')
        return None
    coords, elems = [], []
    for line in lines[2: 2 + n]:
        parts = line.split()
        if len(parts) < 4:
            return None
        elems.append(parts[0])
        coords.append((float(parts[1]), float(parts[2]), float(parts[3])))
    for atom, elem in zip(template_mol.GetAtoms(), elems):
        if atom.GetSymbol() != elem:
            print(f'    Element mismatch: RDKit={atom.GetSymbol()}, xtb={elem}')
            return None
    rw = Chem.RWMol(template_mol)
    rw.RemoveAllConformers()
    conf = Chem.Conformer(n)
    for i, (x, y, z) in enumerate(coords):
        conf.SetAtomPosition(i, rdGeometry.Point3D(x, y, z))
    rw.AddConformer(conf, assignId=True)
    return rw.GetMol()


def polar_sasa(mol, conf_id=0):
    """3D polar SASA using Bondi radii — identical to Tier-1 protocol."""
    try:
        radii = []
        for atom in mol.GetAtoms():
            sym = atom.GetSymbol()
            radii.append(_BONDI.get(sym, 1.50))
            atom.SetIntProp('SASAClass', 0 if sym in _POLAR else 1)
            atom.SetProp('SASAClassName', 'Polar' if sym in _POLAR else 'APolar')
        query = rdFreeSASA.MakeFreeSasaPolarAtomQuery()
        return round(rdFreeSASA.CalcSASA(mol, radii, confIdx=conf_id, query=query), 4)
    except Exception:
        return np.nan


def intramolecular_hbonds(mol, conf_id=0):
    """Count intramolecular H-bonds — identical to Tier-1 protocol."""
    try:
        pos       = mol.GetConformer(conf_id).GetPositions()
        donors    = [i for m in mol.GetSubstructMatches(HB_DONOR)    for i in m]
        acceptors = [i for m in mol.GetSubstructMatches(HB_ACCEPTOR) for i in m]
        count = 0
        for d in donors:
            for h in mol.GetAtomWithIdx(d).GetNeighbors():
                if h.GetAtomicNum() != 1:
                    continue
                h_pos, d_pos = pos[h.GetIdx()], pos[d]
                for a in acceptors:
                    if a == d:
                        continue
                    try:
                        if len(Chem.GetShortestPath(mol, d, a)) < 6:
                            continue
                    except Exception:
                        continue
                    if np.linalg.norm(h_pos - pos[a]) > 3.0:
                        continue
                    vhd = d_pos - h_pos
                    vha = pos[a]  - h_pos
                    cos = np.dot(vhd, vha) / (np.linalg.norm(vhd) * np.linalg.norm(vha) + 1e-9)
                    if np.degrees(np.arccos(np.clip(cos, -1, 1))) >= 120.0:
                        count += 1
        return count
    except Exception:
        return np.nan


def shape_descriptors(mol, conf_id=0):
    try:
        return {
            'Rg':          Descriptors3D.RadiusOfGyration(mol, confId=conf_id),
            'NPR1':        Descriptors3D.NPR1(mol, confId=conf_id),
            'NPR2':        Descriptors3D.NPR2(mol, confId=conf_id),
            'Asphericity': Descriptors3D.Asphericity(mol, confId=conf_id),
        }
    except Exception:
        return {k: np.nan for k in ['Rg', 'NPR1', 'NPR2', 'Asphericity']}


def process_compound(compound):
    mol_id = compound['id']
    smiles = compound['smiles']
    t0     = time.time()

    print(f'  [{mol_id}] Embedding starting geometry...', flush=True)
    work_dir = Path(WORK_ROOT) / str(mol_id)
    work_dir.mkdir(parents=True, exist_ok=True)

    base = {
        'id': mol_id, 'name': compound['name'],
        'pampa': compound['pampa'], 'permeable': compound['permeable'],
        'error': None
    }

    xyz_path, template_mol = smiles_to_xyz(smiles, mol_id, work_dir)
    if xyz_path is None:
        return {**base, 'error': 'embed_failed'}

    results_by_solvent = {}
    for solvent in ('water', 'chcl3'):
        print(f'  [{mol_id}] xtb --opt --gbsa {solvent}...', flush=True)
        opt_xyz = run_xtb_opt(xyz_path, solvent, work_dir)
        if opt_xyz is None:
            return {**base, 'error': f'xtb_failed_{solvent}'}
        mol_out = xyz_to_mol(opt_xyz, template_mol)
        if mol_out is None:
            return {**base, 'error': f'parse_failed_{solvent}'}
        results_by_solvent[solvent] = {
            'psa': polar_sasa(mol_out),
            'hb':  intramolecular_hbonds(mol_out),
            **shape_descriptors(mol_out)
        }

    aq  = results_by_solvent['water']
    mem = results_by_solvent['chcl3']

    def safe_diff(a, b): return float(a - b) if not (np.isnan(a) or np.isnan(b)) else np.nan

    delta_psa = safe_diff(aq['psa'], mem['psa'])
    elapsed   = round(time.time() - t0, 1)
    print(f'  [{mol_id}] Done in {elapsed:.0f}s  '
          f'aq_psa={aq["psa"]:.1f}  mem_psa={mem["psa"]:.1f}  '
          f'ΔPSA={delta_psa:.1f}', flush=True)

    return {
        **base,
        'method':         'xtb-GFN2+GBSA-single-opt',
        'aq_psa3d':       aq['psa'],
        'aq_hb_count':    aq['hb'],
        'aq_Rg':          aq['Rg'],
        'aq_NPR1':        aq['NPR1'],
        'aq_NPR2':        aq['NPR2'],
        'aq_Asphericity': aq['Asphericity'],
        'mem_psa3d':      mem['psa'],
        'mem_hb_count':   mem['hb'],
        'mem_Rg':         mem['Rg'],
        'mem_NPR1':       mem['NPR1'],
        'mem_NPR2':       mem['NPR2'],
        'mem_Asphericity':mem['Asphericity'],
        'delta_psa3d':    delta_psa,
        'delta_hb':       safe_diff(mem['hb'], aq['hb']),
        'delta_Rg':       safe_diff(aq['Rg'], mem['Rg']),
        'wall_s':         elapsed,
        'solvent_aq':     'water (GBSA, eps~80)',
        'solvent_mem':    'chcl3 (GBSA, eps~4.8)',
    }

print('Helper functions loaded.')

In [ ]:
# ── CELL 5: Reference compound definitions ────────────────────────────────────
# SMILES from data/reference_set.csv (canonical, RDKit-standardized).
# Tier-1 ΔPSA from local results/conformer_descriptors_raw.csv (n=20 conformers).

REFERENCE_COMPOUNDS = [
    {
        'id':            'HexPep',
        'name':          'Hexapeptide (c[dL-dL-L-dL-P-Y])',
        'smiles':        'CC(C)C[C@@H]1NC(=O)[C@@H](CC(C)C)NC(=O)[C@@H](CC(C)C)NC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[C@@H]2CCCN2C(=O)[C@@H](CC(C)C)NC1=O',
        'pampa':         -6.2,
        'permeable':     False,
        'tier1_delta_psa': 64.4,   # from conformer_descriptors_raw.csv ID=2
        'db_delta_psa':    2.0,    # CycPeptMPDB H2O_3DPSA - CHCl3_3DPSA
        'lit_delta_psa':   None,
        'lit_source':    'Rezai & Lokey, JACS 2006',
    },
    {
        'id':            '1NMe3',
        'name':          'N-Me Hexapeptide (1NMe3)',
        'smiles':        'CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O)[C@H]2CCCN2C(=O)[C@H](CC(C)C)NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H](CC(C)C)N(C)C1=O',
        'pampa':         -5.52,
        'permeable':     True,
        'tier1_delta_psa': None,   # not in 1500-mol sample; run separately if needed
        'db_delta_psa':    1.0,
        'lit_delta_psa':   None,
        'lit_source':    'White & Lokey, Nat Chem Biol 2011',
    },
    {
        'id':            'CsA',
        'name':          'Cyclosporin A',
        'smiles':        'C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC)C(=O)N(C)CC(=O)N(C)[C@@H](CC(C)C)C(=O)N[C@@H](C(C)C)C(=O)N(C)[C@@H](CC(C)C)C(=O)N[C@@H](C)C(=O)N[C@H](C)C(=O)N(C)[C@@H](CC(C)C)C(=O)N(C)[C@@H](CC(C)C)C(=O)N(C)[C@@H](C(C)C)C(=O)N1C',
        'pampa':         -6.6,
        'permeable':     True,
        'tier1_delta_psa': 84.9,   # from conformer_descriptors_raw.csv ID=1
        'db_delta_psa':    -1.0,   # DB value unreliable (near-zero; see methods)
        'lit_delta_psa':   75.0,   # Witek et al. JCTC 2016 (explicit-solvent MD)
        'lit_source':    'Witek et al., JCTC 2016',
    },
    {
        'id':            'DP172',
        'name':          'DP-172',
        'smiles':        'CC[C@H](C)[C@@H]1NC(=O)[C@H]([C@@H](C)O)NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)N(C)C(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H](C(C)C)NC(=O)C[C@@H](C(=O)N2CCCCC2)NC1=O',
        'pampa':         -4.15,
        'permeable':     True,
        'tier1_delta_psa': 88.9,   # from conformer_descriptors_raw.csv ID=183
        'db_delta_psa':    -47.0,  # DB value anomalous (negative; CHCl3 > H2O)
        'lit_delta_psa':   None,
        'lit_source':    'CHUGAI 2013 pharmaceutical screen',
    },
    {
        'id':            'PSLYF',
        'name':          'c*[PSLYF]',
        'smiles':        'CC(C)C[C@@H]1NC(=O)[C@H](CO)NC(=O)[C@@H]2CCCN2[C@H](C(=O)NC(C)(C)C)[C@H](C)NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H](Cc2ccc(O)cc2)NC1=O',
        'pampa':         -9.1,
        'permeable':     False,
        'tier1_delta_psa': None,   # not in 1500-mol sample
        'db_delta_psa':    0.0,
        'lit_delta_psa':   None,
        'lit_source':    'Hickey, J Med Chem 2016',
    },
]

print(f'Loaded {len(REFERENCE_COMPOUNDS)} reference compounds:')
for c in REFERENCE_COMPOUNDS:
    perm = 'permeable' if c['permeable'] else 'impermeable'
    t1   = f"  Tier-1 ΔPSA={c['tier1_delta_psa']:.1f}" if c['tier1_delta_psa'] else '  Tier-1=—'
    lit  = f"  Lit={c['lit_delta_psa']:.0f} Å²" if c['lit_delta_psa'] else ''
    print(f"  {c['id']:<10}  PAMPA={c['pampa']:.2f}  {perm}{t1}{lit}")

In [ ]:
# ── CELL 6: Run all 5 compounds (sequential, ~3-5 min each) ───────────────────
import pandas as pd
from pathlib import Path

results = []
t_start = time.time()

print(f'Running {len(REFERENCE_COMPOUNDS)} compounds sequentially with xtb GFN2+GBSA...')
print('-' * 65)

for c in REFERENCE_COMPOUNDS:
    mol_id = c['id']
    try:
        result = process_compound(c)
        results.append(result)
        # Save after each compound so Drive has it even if session crashes
        df = pd.DataFrame(results)
        df.to_csv(RESULTS_CSV, index=False)
        status = result.get('error') or 'OK'
        dpsa   = result.get('delta_psa3d', float('nan'))
        wall   = result.get('wall_s', 0)
        print(f'FINISHED: {mol_id:<10}  ΔPSA={dpsa:>7.1f} Å²  t={wall:.0f}s  status={status}')
    except Exception as e:
        print(f'ERROR: {mol_id} raised exception: {e}')
        results.append({'id': mol_id, 'error': str(e)})

total_elapsed = time.time() - t_start
print('-' * 65)
print(f'All done in {total_elapsed/60:.1f} min')
print(f'Results saved to: {RESULTS_CSV}')

In [ ]:
# ── CELL 7: Results summary and comparison table ───────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path

res = pd.read_csv(RESULTS_CSV)
ok  = res[res['error'].isna()].copy()

print(f'Processed: {len(res)}  Successful: {len(ok)}  Failed: {res["error"].notna().sum()}')
if res['error'].notna().any():
    print('Failures:')
    print(res[res['error'].notna()][['id', 'error']].to_string(index=False))

lit_map  = {c['id']: c.get('lit_delta_psa')   for c in REFERENCE_COMPOUNDS}
t1_map   = {c['id']: c.get('tier1_delta_psa') for c in REFERENCE_COMPOUNDS}
db_map   = {c['id']: c.get('db_delta_psa')    for c in REFERENCE_COMPOUNDS}
perm_map = {c['id']: c['permeable']            for c in REFERENCE_COMPOUNDS}

rows = []
for _, r in ok.iterrows():
    cid  = r['id']
    lit  = lit_map.get(cid)
    t1   = t1_map.get(cid)
    db   = db_map.get(cid)
    rows.append({
        'ID':          cid,
        'PAMPA':       r['pampa'],
        'Perm':        'Y' if perm_map.get(cid) else 'N',
        'aq_PSA':      f"{r['aq_psa3d']:.1f}",
        'mem_PSA':     f"{r['mem_psa3d']:.1f}",
        'xtb ΔPSA':    f"{r['delta_psa3d']:.1f}",
        'ΔHB (xtb)':   f"{r['delta_hb']:.0f}",
        'Tier-1 ΔPSA': f"{t1:.1f}" if t1 else '—',
        'DB ΔPSA':     f"{db:.1f}" if db is not None else '—',
        'Lit ΔPSA':    f"~{lit:.0f}" if lit else '—',
    })

table = pd.DataFrame(rows).sort_values('PAMPA', ascending=False)
print('\n' + '='*85)
print('xtb+GBSA Results vs Tier-1 vs DB 3DPSA vs Literature')
print('='*85)
print(table.to_string(index=False))
print('='*85)

# Key finding: permeable vs impermeable ΔPSA
if len(ok) >= 2:
    ok['permeable_bool'] = ok['id'].map(perm_map)
    grp = ok.groupby('permeable_bool')['delta_psa3d'].mean()
    perm_mean = grp.get(True, float('nan'))
    imp_mean  = grp.get(False, float('nan'))
    print(f'\nMean ΔPSA — permeable: {perm_mean:.1f} Å²  |  impermeable: {imp_mean:.1f} Å²')
    if not np.isnan(perm_mean) and not np.isnan(imp_mean):
        diff = perm_mean - imp_mean
        direction = 'higher' if diff > 0 else 'lower'
        consistent = 'CONSISTENT' if diff > 0 else 'INCONSISTENT'
        print(f'→ Permeable compounds show {abs(diff):.1f} Å² {direction} ΔPSA')
        print(f'→ {consistent} with chameleonic hypothesis')

# Save summary
lines = [
    'CHEM 269 — Tier-2 xtb+GBSA Reference Compound Results',
    'Method: GFN2-xTB + GBSA single-structure optimization in water vs CHCl3',
    '',
    table.to_string(index=False),
    '',
    f'Mean ΔPSA permeable: {perm_mean:.1f} Å²  |  impermeable: {imp_mean:.1f} Å²',
]
Path(RESULTS_SUMMARY).write_text('\n'.join(lines))
print(f'\nSummary saved to: {RESULTS_SUMMARY}')

In [ ]:
# ── CELL 8: Download results ───────────────────────────────────────────────────
from google.colab import files
from pathlib import Path

for p in [RESULTS_CSV, RESULTS_SUMMARY]:
    if Path(p).exists():
        sz = Path(p).stat().st_size / 1e3
        print(f'Downloading {Path(p).name} ({sz:.1f} KB)...')
        files.download(p)
    else:
        print(f'Not found: {p}')